# 3.2 线性回归的从零开始实现

本节只借助 PyTorch 的张量与自动微分功能，不使用 `nn.Linear`、内置损失函数或优化器，完整实现线性回归的训练流程。

学习目标：

1. 人工生成带噪声的线性数据；
2. 手写小批量数据迭代器；
3. 手写线性模型、平方损失和小批量随机梯度下降；
4. 理解参数初始化、反向传播、梯度清零和无梯度更新；
5. 检查训练结果是否恢复了真实参数。

对应教材：[3.2 线性回归的从零开始实现](https://zh.d2l.ai/chapter_linear-networks/linear-regression-scratch.html)

In [1]:
import random
import torch

# 固定随机种子，使本 notebook 的结果更容易复现
random.seed(42)
torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.13.0+cpu


## 3.2.1 生成数据集

假设样本特征 $\mathbf{x}\in\mathbb{R}^2$ 从标准正态分布中独立采样，标签由真实线性模型生成：

$$
y=\mathbf{x}^{\top}\mathbf{w}+b+\epsilon,
$$

其中真实参数为 $\mathbf{w}=[2,-3.4]^{\top}$、$b=4.2$，噪声 $\epsilon\sim\mathcal{N}(0,0.01^2)$。训练的目标是仅根据特征和标签，重新估计出这些参数。

In [2]:
def synthetic_data(w, b, num_examples):
    """生成 y = Xw + b + 高斯噪声。"""
    X = torch.normal(0, 1, (num_examples, len(w)))
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape((-1, 1))

true_w = torch.tensor([2.0, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)

print("features shape:", features.shape)
print("labels shape:  ", labels.shape)
print("第一个样本:", features[0], "->", labels[0].item())

features shape: torch.Size([1000, 2])
labels shape:   torch.Size([1000, 1])
第一个样本: tensor([1.9269, 1.4873]) -> 2.9871368408203125


## 3.2.2 读取数据集

一次使用全部样本计算梯度可能很慢。小批量随机梯度下降会先打乱样本索引，再按 `batch_size` 分组。每轮完整访问训练集一次称为一个 **epoch**。

最后一个批次的样本数可能小于 `batch_size`，因此切片时不应假定每批大小固定。

In [3]:
def data_iter(batch_size, features, labels):
    """随机打乱数据，并逐批返回特征和标签。"""
    num_examples = len(features)
    indices = list(range(num_examples))
    random.shuffle(indices)
    for i in range(0, num_examples, batch_size):
        batch_indices = torch.tensor(indices[i:i + batch_size])
        yield features[batch_indices], labels[batch_indices]

batch_size = 10
X_batch, y_batch = next(iter(data_iter(batch_size, features, labels)))
print("一个小批量的形状:", X_batch.shape, y_batch.shape)

一个小批量的形状: torch.Size([10, 2]) torch.Size([10, 1])


## 3.2.3 初始化模型参数与定义模型

权重使用均值为 0、标准差为 0.01 的正态分布初始化，偏置初始化为 0。设置 `requires_grad=True` 后，PyTorch 会记录与参数有关的运算，以便自动求导。

模型的批量形式为

$$
\hat{\mathbf y}=\mathbf X\mathbf w+b.
$$

In [4]:
w = torch.normal(0, 0.01, size=(2, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)

def linreg(X, w, b):
    """线性回归模型。"""
    return torch.matmul(X, w) + b

print("初始 w:", w.detach().reshape(-1).tolist())
print("初始 b:", b.item())

初始 w: [0.01142729539424181, -0.0007455555605702102]
初始 b: 0.0


## 3.2.4 定义损失函数

对每个样本使用平方损失：

$$
l^{(i)}(\mathbf w,b)=\frac{1}{2}\left(\hat y^{(i)}-y^{(i)}\right)^2.
$$

保留每个样本的损失，训练时再通过 `.sum()` 汇总并反向传播。优化器会除以实际批量大小，因此参数更新对应小批量平均梯度。

In [5]:
def squared_loss(y_hat, y):
    """逐样本平方损失。"""
    return (y_hat - y.reshape(y_hat.shape)) ** 2 / 2

## 3.2.5 定义优化算法

小批量随机梯度下降的更新规则为

$$
\theta\leftarrow\theta-\frac{\eta}{|\mathcal B|}\nabla_{\theta}\sum_{i\in\mathcal B}l^{(i)}.
$$

`torch.no_grad()` 表示参数更新本身不需要加入计算图。更新完成后必须清空梯度，因为 PyTorch 默认会累加梯度。

In [6]:
def sgd(params, lr, batch_size):
    """小批量随机梯度下降。"""
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()

## 3.2.6 训练

每个 epoch 执行四个步骤：

1. 前向传播，得到预测值；
2. 计算当前小批量的损失；
3. 反向传播，计算参数梯度；
4. 使用 SGD 更新参数并清空梯度。

这里使用学习率 0.03，训练 3 个 epoch。对这一简单的凸优化问题，损失应快速下降到接近噪声水平。

In [7]:
lr = 0.03
num_epochs = 3

for epoch in range(num_epochs):
    for X, y in data_iter(batch_size, features, labels):
        loss = squared_loss(linreg(X, w, b), y)
        loss.sum().backward()
        sgd([w, b], lr, len(X))

    with torch.no_grad():
        train_loss = squared_loss(linreg(features, w, b), labels).mean()
    print(f"epoch {epoch + 1}, loss {train_loss.item():.8f}")

epoch 1, loss 0.04718711
epoch 2, loss 0.00019885
epoch 3, loss 0.00005020


## 3.2.7 评估结果

因为这是人工数据，我们知道真实参数，可以直接计算估计误差。在真实任务中通常不知道真实参数，应改用独立验证集或测试集上的预测指标评价模型。

In [8]:
with torch.no_grad():
    w_error = true_w - w.reshape(true_w.shape)
    b_error = true_b - b.item()

print("真实 w:", true_w.tolist())
print("学得 w:", w.detach().reshape(-1).tolist())
print("w 误差:", w_error.tolist())
print(f"真实 b: {true_b:.4f}")
print(f"学得 b: {b.item():.4f}")
print(f"b 误差: {b_error:.6f}")

# 在正常运行环境中，这两个断言应通过
assert torch.max(torch.abs(w_error)) < 0.1
assert abs(b_error) < 0.1

真实 w: [2.0, -3.4000000953674316]
学得 w: [1.9994313716888428, -3.399848699569702]
w 误差: [0.0005686283111572266, -0.0001513957977294922]
真实 b: 4.2000
学得 b: 4.1999
b 误差: 0.000145


## 易错点

- **形状不一致**：标签应为 `(batch_size, 1)`，避免与 `(batch_size,)` 广播成二维矩阵。
- **忘记反向传播**：没有调用 `loss.sum().backward()` 就不会产生梯度。
- **忘记清空梯度**：PyTorch 默认累加 `param.grad`，会导致更新越来越异常。
- **在计算图中更新参数**：参数更新应放在 `torch.no_grad()` 中。
- **固定除以设定的批量大小**：最后一批可能更小，更新时应使用 `len(X)`。
- **学习率不合适**：过大会使损失震荡或发散，过小则收敛缓慢。

## 小结

- 一个完整训练流程包含数据、模型、损失函数和优化算法四个核心部分。
- 小批量随机梯度下降在每轮中随机抽取小批量，用其平均梯度近似总体梯度。
- 自动微分负责计算梯度，但参数更新与梯度清零仍由我们显式实现。
- 线性回归虽然简单，却完整展示了后续深度学习模型共同使用的训练范式。

## 练习

1. 将样本数从 1000 改为 100，参数估计误差如何变化？为什么？
2. 将噪声标准差从 0.01 改为 0.1 或 1.0，观察训练损失和参数误差。
3. 尝试学习率 `0.003`、`0.03`、`0.3` 和 `3.0`，比较收敛速度与稳定性。
4. 把批量大小改为 1、10、100 和 1000，记录每个 epoch 的损失。
5. 如果把 `param.grad.zero_()` 删除，会发生什么？解释梯度累加带来的影响。
6. 修改数据生成过程，使特征数从 2 增加到 5，并验证代码是否仍能恢复真实参数。
7. 不使用自动微分，依据平方损失手工推导并实现 $\mathbf w$ 与 $b$ 的梯度。

### 思考：手工梯度

对一个小批量，令 $\mathbf e=\mathbf X\mathbf w+b-\mathbf y$，则平均平方损失对应的梯度为

$$
\nabla_{\mathbf w}L=\frac{1}{|\mathcal B|}\mathbf X^{\top}\mathbf e,
\qquad
\frac{\partial L}{\partial b}=\frac{1}{|\mathcal B|}\sum_i e_i.
$$

可以将这两个表达式与自动微分得到的 `w.grad / len(X)`、`b.grad / len(X)` 对比。